## install libraries

In [1]:
pip install tensorflow matplotlib

  Using cached wheel-0.46.3-py3-none-any.whl.metadata (2.4 kB)
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 200.7/200.7 MB 2.1 MB/s eta 0:00:0000:0100:03
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 11.8/11.8 MB 1.6 MB/s eta 0:00:0000:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 676.9/676.9 kB 2.0 MB/s eta 0:00:00-:--:--
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 5.5/5.5 MB 2.2 MB/s eta 0:00:00a 0:00:01
Using cached wheel-0.46.3-py3-none-any.whl (30 kB)
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.8/2.8 MB 1.6 MB/s eta 0:00:00a 0:00:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.5/1.5 MB 2.5 MB/s eta 0:00:00a 0:00:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 25.8/25.8 MB 2.4 MB/s eta 0:00:0000:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.0/1.0 MB 3.9 MB/s eta 0:00:00a 0:00:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 24/24 [tensorflow]4 [tensorflow]]

[notice] A new release of pip is available: 25.1.1 -> 26.0.1
[notice] To update, run: pip3 install --upgrade pip
N

## import libraries

In [2]:
import tensorflow as tf
from tensorflow.keras import layers, models
import matplotlib.pyplot as plt

## Load Dataset from Folders

In [12]:
img_size = 48
batch_size = 64

train_ds = tf.keras.utils.image_dataset_from_directory(
    "Datasets/FER-2013/train",
    validation_split=0.2,
    subset="training",
    seed=123,
    image_size=(img_size, img_size),
    batch_size=batch_size,
    color_mode="grayscale"
)


test_ds = tf.keras.utils.image_dataset_from_directory(
    "Datasets/FER-2013/test",
    image_size=(img_size, img_size),
    batch_size=batch_size,
    color_mode="grayscale"
)

# SAVE CLASS INFO HERE
class_names = train_ds.class_names
num_classes = len(class_names)

print("Classes:", class_names)

Found 29172 files belonging to 7 classes.
Using 23338 files for training.
Found 7178 files belonging to 7 classes.
Classes: ['angry', 'disgust', 'fear', 'happy', 'neutral', 'sad', 'surprise']


## Normalize Images

In [13]:
normalization_layer = layers.Rescaling(1./255)

train_ds = train_ds.map(lambda x, y: (normalization_layer(x), y))
test_ds  = test_ds.map(lambda x, y: (normalization_layer(x), y))

## Build CNN Model

In [14]:
num_classes = len(class_names)

model = models.Sequential([

    layers.Conv2D(32, (3,3), activation='relu', input_shape=(48,48,1)),
    layers.BatchNormalization(),
    layers.MaxPooling2D(2,2),

    layers.Conv2D(64, (3,3), activation='relu'),
    layers.BatchNormalization(),
    layers.MaxPooling2D(2,2),

    layers.Conv2D(128, (3,3), activation='relu'),
    layers.BatchNormalization(),
    layers.MaxPooling2D(2,2),

    layers.GlobalAveragePooling2D(),

    layers.Dense(128, activation='relu'),
    layers.Dropout(0.5),

    layers.Dense(num_classes, activation='softmax')
])

/Library/Frameworks/Python.framework/Versions/3.13/lib/python3.13/site-packages/keras/src/layers/convolutional/base_conv.py:113: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(activity_regularizer=activity_regularizer, **kwargs)


## Compile

In [15]:
model.compile(
    optimizer='adam',
    loss='sparse_categorical_crossentropy',
    metrics=['accuracy']
)

## Train

In [16]:
from sklearn.utils.class_weight import compute_class_weight
import numpy as np

all_labels = []

for _, y in train_ds.unbatch():
    all_labels.append(y.numpy())

all_labels = np.array(all_labels)

class_weights = compute_class_weight(
    class_weight="balanced",
    classes=np.unique(all_labels),
    y=all_labels
)

class_weights = dict(enumerate(class_weights))

print("Class Weights:", class_weights)

Class Weights: {0: np.float64(0.9289495681248259), 1: np.float64(9.525714285714285), 2: np.float64(1.0099969706149652), 3: np.float64(0.5761188871608778), 4: np.float64(0.8513789581205311), 5: np.float64(0.8623900672529746), 6: np.float64(1.3183076314748912)}


2026-02-26 20:25:40.248634: I tensorflow/core/framework/local_rendezvous.cc:407] Local rendezvous is aborting with status: OUT_OF_RANGE: End of sequence


In [17]:
from tensorflow.keras.callbacks import EarlyStopping
from tensorflow.keras.callbacks import ReduceLROnPlateau

early_stop = EarlyStopping(
    monitor='val_loss',
    patience=5,
    restore_best_weights=True
)

lr_reduce = ReduceLROnPlateau(
    monitor='val_loss',
    factor=0.5,
    patience=3,
    min_lr=1e-6,
    verbose=1
)

model.fit(
    train_ds,
    validation_data=test_ds,
    epochs=50,
    callbacks=[early_stop, lr_reduce],
    class_weight=class_weights
)
model.evaluate(test_ds)

Epoch 1/50
365/365 ━━━━━━━━━━━━━━━━━━━━ 35s 89ms/step - accuracy: 0.1944 - loss: 1.9516 - val_accuracy: 0.1482 - val_loss: 2.0370 - learning_rate: 0.0010
Epoch 2/50
365/365 ━━━━━━━━━━━━━━━━━━━━ 29s 78ms/step - accuracy: 0.2726 - loss: 1.7741 - val_accuracy: 0.2165 - val_loss: 1.9113 - learning_rate: 0.0010
Epoch 3/50
365/365 ━━━━━━━━━━━━━━━━━━━━ 29s 78ms/step - accuracy: 0.3422 - loss: 1.6536 - val_accuracy: 0.2905 - val_loss: 1.7521 - learning_rate: 0.0010
Epoch 4/50
365/365 ━━━━━━━━━━━━━━━━━━━━ 30s 82ms/step - accuracy: 0.3969 - loss: 1.5521 - val_accuracy: 0.3594 - val_loss: 1.6571 - learning_rate: 0.0010
Epoch 5/50
365/365 ━━━━━━━━━━━━━━━━━━━━ 30s 83ms/step - accuracy: 0.4341 - loss: 1.4585 - val_accuracy: 0.3880 - val_loss: 1.5969 - learning_rate: 0.0010
Epoch 6/50
365/365 ━━━━━━━━━━━━━━━━━━━━ 30s 82ms/step - accuracy: 0.4566 - loss: 1.3780 - val_accuracy: 0.3968 - val_loss: 1.5595 - learning_rate: 0.0010
Epoch 7/50
365/365 ━━━━━━━━━━━━━━━━━━━━ 31s 84ms/step - accuracy: 0.4842 - l

[1.2691831588745117, 0.545834481716156]

In [18]:
print(class_names)

['angry', 'disgust', 'fear', 'happy', 'neutral', 'sad', 'surprise']


## Save model

In [21]:
model.save("emotion_model.h5")

In [20]:
import os

train_path = "/Users/mayanksmac/Desktop/DL/Datasets/FER-2013/train"

for cls in sorted(os.listdir(train_path)):
    cls_path = os.path.join(train_path, cls)

    if os.path.isdir(cls_path):   # ← Ignore .DS_Store
        print(cls, len(os.listdir(cls_path)))

angry 4458
disgust 436
fear 4097
happy 7215
neutral 4965
sad 4830
surprise 3171


In [1]:
pip install PyQt5

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 6.6/6.6 MB 3.1 MB/s eta 0:00:00a 0:00:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 36.8/36.8 MB 2.9 MB/s eta 0:00:0000:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3/3 [PyQt5]32m2/3 [PyQt5]

[notice] A new release of pip is available: 25.1.1 -> 26.0.1
[notice] To update, run: pip3 install --upgrade pip
Note: you may need to restart the kernel to use updated packages.


In [ ]:
pip uninstall av-0.8.0 -y

Found existing installation: av 16.1.0
Uninstalling av-16.1.0:
  Would remove:
    /Library/Frameworks/Python.framework/Versions/3.13/bin/pyav
    /Library/Frameworks/Python.framework/Versions/3.13/lib/python3.13/site-packages/av-16.1.0.dist-info/*
    /Library/Frameworks/Python.framework/Versions/3.13/lib/python3.13/site-packages/av/*
Proceed (Y/n)? 

In [ ]:
y

In [1]:
pip install speechrecognition

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 32.9/32.9 MB 532.9 kB/s eta 0:00:0000:0100:02

[notice] A new release of pip is available: 25.1.1 -> 26.0.1
[notice] To update, run: pip3 install --upgrade pip
Note: you may need to restart the kernel to use updated packages.
